In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import json
from collections import Counter
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.models import DenseNet121_Weights

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.preprocessing import LabelEncoder

# Set seeds
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
set_seed(42)

# ==========================================
# CONFIGURATION
# ==========================================
DATA_DIR = '../data'
MODEL_DIR = '../models'
os.makedirs(MODEL_DIR, exist_ok=True)

IMAGE_SIZE = 224
BATCH_SIZE = 64
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4
NUM_CLASSES = 7
PATIENCE = 10 # Early stopping
NUM_WORKERS = 2
PIN_MEMORY = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
import torch

print("=" * 50)
print("GPU DIAGNOSTIC")
print("=" * 50)

# 1. Basic CUDA availability
print(f"\n1. CUDA available (basic check): {torch.cuda.is_available()}")

# 2. CUDA version PyTorch was built with
print(f"2. PyTorch CUDA version: {torch.version.cuda}")

# 3. Number of GPUs detected
print(f"3. GPU count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    # 4. GPU name
    print(f"4. GPU name: {torch.cuda.get_device_name(0)}")
    
    # 5. GPU memory
    total_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"5. GPU memory: {total_mem:.1f} GB")
    
    # 6. Quick tensor transfer test
    x = torch.randn(1000, 1000)
    x_gpu = x.to('cuda')
    print(f"6. Tensor transfer to GPU: ✓ Works")
else:
    print("\n⚠️  NO GPU DETECTED")
    print("Possible reasons:")
    print("  - You don't have an NVIDIA GPU")
    print("  - NVIDIA drivers not installed")
    print("  - You installed PyTorch CPU-only version")
    print("\nTo install PyTorch with CUDA, run:")
    print("  pip uninstall torch torchvision")
    print("  pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118")

GPU DIAGNOSTIC

1. CUDA available (basic check): False
2. PyTorch CUDA version: None
3. GPU count: 0

⚠️  NO GPU DETECTED
Possible reasons:
  - You don't have an NVIDIA GPU
  - NVIDIA drivers not installed
  - You installed PyTorch CPU-only version

To install PyTorch with CUDA, run:
  pip uninstall torch torchvision
  pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118


In [1]:
import torch
print(f"Version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Version: 2.13.0+cu126
CUDA available: True


In [4]:
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Try to load saved splits
if os.path.exists(f'{DATA_DIR}/train_split.csv'):
    train_df = pd.read_csv(f'{DATA_DIR}/train_split.csv')
    val_df = pd.read_csv(f'{DATA_DIR}/val_split.csv')
    test_df = pd.read_csv(f'{DATA_DIR}/test_split.csv')
    print("Loaded saved splits.")
else:
    print("Split files not found. Generating splits now...")
    # Load full metadata
    df = pd.read_csv(f'{DATA_DIR}/HAM10000_metadata.csv')
    df['path'] = df['image_id'].apply(lambda x: f'{DATA_DIR}/{x}.jpg')
    df = df[df['path'].apply(os.path.exists)].reset_index(drop=True)
    
    le = LabelEncoder()
    df['label'] = le.fit_transform(df['dx'])
    
    # Stratified splits
    train_val_df, test_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=42)
    train_df, val_df = train_test_split(train_val_df, test_size=0.15, stratify=train_val_df['label'], random_state=42)
    
    # Save for future use
    train_df.to_csv(f'{DATA_DIR}/train_split.csv', index=False)
    val_df.to_csv(f'{DATA_DIR}/val_split.csv', index=False)
    test_df.to_csv(f'{DATA_DIR}/test_split.csv', index=False)
    print("Splits generated and saved.")

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Split files not found. Generating splits now...
Splits generated and saved.
Train: 7235 | Val: 1277 | Test: 1503


In [16]:
from PIL import Image
# Training transforms
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=25),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation/Test transforms
val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Dataset class
class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx]['path']
        label = self.dataframe.iloc[idx]['label']
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

# Create datasets
train_dataset = SkinLesionDataset(train_df, transform=train_transforms)
val_dataset = SkinLesionDataset(val_df, transform=val_transforms)
test_dataset = SkinLesionDataset(test_df, transform=val_transforms)

# Weighted sampler
class_counts = Counter(train_df['label'])
weights = [1.0 / class_counts[l] for l in train_df['label']]
sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(weights),
    num_samples=len(weights),
    replacement=False
)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("DataLoaders ready.")

DataLoaders ready.


In [25]:
class DermAIdModel(nn.Module):
    """DenseNet-121 with custom classification head."""
    
    def __init__(self, num_classes=7, dropout_rate=0.3):
        super(DermAIdModel, self).__init__()
        
        self.backbone = models.densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
        
        # Start with all layers frozen - we'll unfreeze gradually
        for param in self.backbone.parameters():
            param.requires_grad = False
        
        in_features = self.backbone.classifier.in_features
        
        # Simpler, stronger classification head
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )
        
    def forward(self, x, return_features=False):
        features = self.backbone.features(x)
        features = nn.functional.adaptive_avg_pool2d(features, (1, 1))
        features = torch.flatten(features, 1)
        logits = self.classifier(features)
        
        if return_features:
            return logits, features
        return logits
    
    def unfreeze_backbone(self):
        """Gradually unfreeze backbone for fine-tuning."""
        for param in self.backbone.parameters():
            param.requires_grad = True
        print("Backbone unfrozen for fine-tuning.")
    
    def enable_monte_carlo_dropout(self):
        """Enable dropout at inference for uncertainty."""
        for module in self.modules():
            if isinstance(module, nn.Dropout):
                module.train()

model = DermAIdModel(num_classes=NUM_CLASSES, dropout_rate=0.3)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters (head only): {trainable_params:,}")
print(f"Device: {device}")

Total parameters: 8,638,319
Trainable parameters (head only): 659,463
Device: cuda


In [18]:
# Load label mapping
import json

with open(f'{MODEL_DIR}/label_mapping.json', 'r') as f:
    label_mapping = json.load(f)

idx_to_label = label_mapping['idx_to_label']
class_names = label_mapping['class_names']

print("Label mapping loaded:")
for i in range(7):
    label_code = idx_to_label[str(i)]
    print(f"  {i}: {label_code} - {class_names[label_code]}")

Label mapping loaded:
  0: akiec - Actinic Keratosis
  1: bcc - Basal Cell Carcinoma
  2: bkl - Benign Keratosis
  3: df - Dermatofibroma
  4: mel - Melanoma
  5: nv - Melanocytic Nevus
  6: vasc - Vascular Lesion


In [19]:
import os
import json

os.makedirs(MODEL_DIR, exist_ok=True)

# The 7 skin lesion classes
class_names = {
    'akiec': 'Actinic Keratosis',
    'bcc': 'Basal Cell Carcinoma',
    'bkl': 'Benign Keratosis',
    'df': 'Dermatofibroma',
    'mel': 'Melanoma',
    'nv': 'Melanocytic Nevus',
    'vasc': 'Vascular Lesion'
}

# Create mapping
idx_to_label = {str(i): code for i, code in enumerate(class_names.keys())}
label_to_idx = {code: i for i, code in enumerate(class_names.keys())}

label_mapping = {
    'label_to_idx': label_to_idx,
    'idx_to_label': idx_to_label,
    'class_names': class_names
}

with open(f'{MODEL_DIR}/label_mapping.json', 'w') as f:
    json.dump(label_mapping, f, indent=2)

print("Label mapping saved.")

# Now set the variables for use
idx_to_label = label_mapping['idx_to_label']
class_names = label_mapping['class_names']

for i in range(7):
    label_code = idx_to_label[str(i)]
    print(f"  {i}: {label_code} - {class_names[label_code]}")

Label mapping saved.
  0: akiec - Actinic Keratosis
  1: bcc - Basal Cell Carcinoma
  2: bkl - Benign Keratosis
  3: df - Dermatofibroma
  4: mel - Melanoma
  5: nv - Melanocytic Nevus
  6: vasc - Vascular Lesion


In [20]:
# Load label mapping
import json

with open(f'{MODEL_DIR}/label_mapping.json', 'r') as f:
    label_mapping = json.load(f)

idx_to_label = label_mapping['idx_to_label']
class_names = label_mapping['class_names']

print("Label mapping loaded:")
for i in range(7):
    label_code = idx_to_label[str(i)]
    print(f"  {i}: {label_code} - {class_names[label_code]}")

Label mapping loaded:
  0: akiec - Actinic Keratosis
  1: bcc - Basal Cell Carcinoma
  2: bkl - Benign Keratosis
  3: df - Dermatofibroma
  4: mel - Melanoma
  5: nv - Melanocytic Nevus
  6: vasc - Vascular Lesion


In [21]:
# Calculate class weights (inverse frequency, normalized)
class_weights = []
for i in range(NUM_CLASSES):
    count = class_counts.get(i, 1)
    class_weights.append(1.0 / count)

class_weights = torch.FloatTensor(class_weights).to(device)
# Normalize
class_weights = class_weights / class_weights.sum() * NUM_CLASSES

print("Class weights for loss function:")
for i in range(NUM_CLASSES):
    print(f"  {idx_to_label[str(i)]} ({class_names[idx_to_label[str(i)]]}): {class_weights[i]:.4f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)

Class weights for loss function:
  akiec (Actinic Keratosis): 0.9420
  bcc (Basal Cell Carcinoma): 0.5976
  bkl (Benign Keratosis): 0.2800
  df (Dermatofibroma): 2.6785
  mel (Melanoma): 0.2765
  nv (Melanocytic Nevus): 0.0459
  vasc (Vascular Lesion): 2.1795


In [22]:
# Optimizer - only trainable params
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

# Scheduler - reduce LR when validation loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,
    min_lr=1e-7
)

# Optional: Cosine annealing for final fine-tuning
# scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

In [23]:
def train_epoch(model, loader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc="Training", leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Track metrics
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        # Update progress bar
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    
    return epoch_loss, epoch_acc


def validate(model, loader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating", leave=False):
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    
    return epoch_loss, epoch_acc, all_preds, all_labels, all_probs

In [26]:
def train_phase(model, train_loader, val_loader, criterion, optimizer, scheduler, 
                num_epochs, device, patience, model_dir, phase_name="Phase"):
    """Training loop for one phase."""
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
    best_val_loss = float('inf')
    patience_counter = 0
    
    print(f"\n{'='*60}")
    print(f"{phase_name}")
    print(f"{'='*60}")
    
    for epoch in range(num_epochs):
        epoch_start = time.time()
        
        # Train
        model.train()
        running_loss = 0.0
        all_preds, all_labels = [], []
        
        pbar = tqdm(train_loader, desc=f"{phase_name} Epoch {epoch+1}", leave=False)
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            pbar.set_postfix({'loss': f'{loss.item():.3f}'})
        
        train_loss = running_loss / len(train_loader.dataset)
        train_acc = accuracy_score(all_labels, all_preds)
        
        # Validate
        model.eval()
        val_running_loss = 0.0
        val_preds, val_labels = [], []
        
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc="Validating", leave=False):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_running_loss += loss.item() * images.size(0)
                _, preds = torch.max(outputs, 1)
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
        
        val_loss = val_running_loss / len(val_loader.dataset)
        val_acc = accuracy_score(val_labels, val_preds)
        current_lr = optimizer.param_groups[0]['lr']
        
        # Store history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)
        
        # Scheduler step
        scheduler.step(val_loss)
        
        print(f"Epoch {epoch+1}/{num_epochs} | Time: {time.time()-epoch_start:.1f}s")
        print(f"  Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
        print(f"  Val Loss:   {val_loss:.4f} | Acc: {val_acc:.4f} | LR: {current_lr:.2e}")
        
        # Checkpoint
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'val_acc': val_acc,
            }, f'{model_dir}/best_model.pth')
            print(f"  ✓ Best model saved! (Val Loss: {val_loss:.4f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  ⏹ Early stopping at epoch {epoch+1}")
                break
    
    # Load best model
    checkpoint = torch.load(f'{model_dir}/best_model.pth', map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Best val acc: {checkpoint['val_acc']:.4f}")
    
    return model, history


# ==========================================
# PHASE 1: Train Only the Classification Head
# ==========================================
print("\n" + "="*60)
print("PHASE 1: Training Classification Head (Backbone Frozen)")
print("="*60)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-7)

model, history_phase1 = train_phase(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    num_epochs=30, device=device, patience=8, model_dir=MODEL_DIR,
    phase_name="PHASE 1"
)

# ==========================================
# PHASE 2: Fine-Tune Entire Network
# ==========================================
print("\n" + "="*60)
print("PHASE 2: Fine-Tuning Full Network")
print("="*60)

model.unfreeze_backbone()

# Lower learning rate for fine-tuning
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-8)

model, history_phase2 = train_phase(
    model, train_loader, val_loader, criterion, optimizer, scheduler,
    num_epochs=30, device=device, patience=10, model_dir=MODEL_DIR,
    phase_name="PHASE 2"
)

# Combine histories for plotting
history = {
    'train_loss': history_phase1['train_loss'] + history_phase2['train_loss'],
    'train_acc': history_phase1['train_acc'] + history_phase2['train_acc'],
    'val_loss': history_phase1['val_loss'] + history_phase2['val_loss'],
    'val_acc': history_phase1['val_acc'] + history_phase2['val_acc'],
    'lr': history_phase1['lr'] + history_phase2['lr'],
}

print("\n" + "="*60)
print("TRAINING COMPLETE")
print(f"Best validation accuracy: {max(history['val_acc']):.4f}")
print("="*60)


PHASE 1: Training Classification Head (Backbone Frozen)

PHASE 1


PHASE 1 Epoch 1:   0%|          | 0/114 [00:00<?, ?it/s]

Validating:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 1/30 | Time: 114.2s
  Train Loss: 0.9075 | Acc: 0.5982
  Val Loss:   6.4730 | Acc: 0.6695 | LR: 1.00e-03
  ✓ Best model saved! (Val Loss: 6.4730)


PHASE 1 Epoch 2:   0%|          | 0/114 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 23/30 | Time: 309.9s
  Train Loss: 0.4543 | Acc: 0.8680
  Val Loss:   1.4047 | Acc: 0.8019 | LR: 5.00e-06


PHASE 2 Epoch 24:   0%|          | 0/114 [00:00<?, ?it/s]

Validating:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 24/30 | Time: 310.9s
  Train Loss: 0.4487 | Acc: 0.8648
  Val Loss:   1.2262 | Acc: 0.8246 | LR: 5.00e-06
  ⏹ Early stopping at epoch 24
Best val acc: 0.8136

TRAINING COMPLETE
Best validation accuracy: 0.8246
